In [3]:
# Import the libraries
import pandas as pd
import numpy as np
import random

# Load the messy data
df = pd.read_csv(r"C:\Users\HomePC\Downloads\marketing_campaign_data_messy.csv")

# Print some info. about the data like the shape for the rows and the shape for the columns
print(f"Loadad Dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df

Loadad Dataset: 2020 rows, 12 columns


,Campaign_ID,Campaign_Name,Start_Date,End_Date,Channel,Impressions,Clicks,Spend,Conversions,Active,Clicks,Campaign_Tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24 00:00:00,2023-12-13,TikTok,16795,197,$102.82,20.0,Y,NaN,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06 00:00:00,2023-05-12,Facebook,1860,30,24.33,1.0,0,NaN,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13 00:00:00,2023-12-20,Email,77820,843,1323.39,51.0,No,NaN,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,2023-10-30,2023-11-03,TikTok,55886,2019,2180.38,135.0,True,NaN,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22 00:00:00,2023-04-23,Facebook,7265,169,252.44,30.0,Yes,NaN,FA
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31 00:00:00,2023-11-13,TikTok,30592,586,$503.95,77.0,1,NaN,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01 00:00:00,2023-09-26,Google Ads,20097,897,1641.0,162.0,0,NaN,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09 00:00:00,2023-02-21,Instagram,33254,1117,883.82,214.0,0,NaN,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30 00:00:00,2023-04-27,Facebook,68728,2960,4198.5,591.0,Yes,NaN,FA


## STEP 1: CLEANING HEADERS

In [4]:
# STEP 1: CLEANING HEADERS

# Print the columns and moving the into a list. In some cases, there are spaces before anf after the column names.
print(df.columns.tolist())

# Selecting all the coumns in the dataset as strings and stripping all the spaces at the start and end of column names. Also converting the column names to small letters.
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')


print("FIX APPLIED") # The name of the initial columns
print(df.columns.tolist()) # The new columns

[' Campaign_ID ', 'Campaign_Name', 'Start_Date', 'End_Date', 'Channel', 'Impressions', 'Clicks ', 'Spend', 'Conversions', 'Active', 'Clicks', 'Campaign_Tag']
FIX APPLIED
['campaign_id', 'campaign_name', 'start_date', 'end_date', 'channel', 'impressions', 'clicks', 'spend', 'conversions', 'active', 'clicks', 'campaign_tag']


## STEP 2: TYPE CONVERSION & CURRENCY CLEANING

In [5]:
# STEP 2: TYPE CONVERSION & CURRENCY CLEANING

# Define a column dirty_spend_mask where the 'spend column contains \ and $
dirty_spend_mask = df['spend'].astype(str).str.contains(r'\$')
print(df.loc[dirty_spend_mask,['campaign_id','spend']].head(3))

df['spend'] = df['spend'].astype(str).str.replace(r'[^\d.-]', '', regex=True)
df['spend'] = pd.to_numeric(df['spend'])

print("FIX APPLIED")
print(df.loc[dirty_spend_mask,['campaign_id','spend']].head(3))

   campaign_id     spend
0    CMP-00001   $102.82
21   CMP-00022   $2428.4
22   CMP-00023  $4726.22
FIX APPLIED
   campaign_id    spend
0    CMP-00001   102.82
21   CMP-00022  2428.40
22   CMP-00023  4726.22


## STEP 3: CATEGORICAL TYPOS (FUZZY LOGIC)

In [6]:
# STEP 3: CATEGORICAL TYPOS (FUZZY LOGIC)

# Printing the unique values from the 'channel' column
print(df['channel'].unique()) # There are many  different values in the 'channel' column that should be meerged together

# Creating a cleanup map to merged the different spelt names
cleanup_map = {
    'Facebok': 'Facebook',
    'Insta_gram': 'Instagram',
    'Gogle': 'Google_Ads',
    'Tik_Tok': 'TikTok',
    'E-mail': 'Email',
    'N/A': np.nan # Handling the ghost value here too
}

# Redifining df_channel. Replacing the values in the channel with the cleaunup_map
df['channel'] = df['channel'].replace(cleanup_map)

print('FIX APPLIED')
print(df['channel'].unique())

['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' 'E-mail' nan 'Gogle'
 'Tik_Tok' 'Facebok' 'Insta_gram']
FIX APPLIED
['TikTok' 'Facebook' 'Email' 'Instagram' 'Google Ads' nan 'Google_Ads']


## STEP 4: HANDLING MIXED BOOLEANS

Values like True/False, Yes/No, 0/1

In [7]:
# STEP 4: HANDLING MIXED BOOLEANS

# Print what's in the 'active' column
print(df['active'].unique())

# cleanup map to merged the duplicate values and setting 'Yes', 'Y', '1', and 1 to 'True' as well as 'No', '0', and 0 to 'False'.
bool_map = {
    'Yes': 'True',
    'Y': 'True',
    '1': 'True',
    1 : 'True',
    'No': 'False',
    '0': 'False',
    0 : 'False'
}

# Redifining the df. If there are na values, it's going to be filled with 'False'
df['active'] = df['active'].map(bool_map).fillna(False).astype(bool)

print('FIX APPLIED')
print(df['active'].unique())

['Y' '0' 'No' 'True' 'Yes' '1' 'False']
FIX APPLIED
[ True False]


## STEP 5: DATE PARSING

In [8]:
# STEP 5: DATE PARSING

# Printing the data type for the start date column
print(df['start_date'].dtype) # The datatype for the 'start_date' column is an object. It needs to be made a date datatype

# Changing the datatype for the 'start_datae' and 'end_date' using Pandas. If there are errors or things that can't be converted, 'coerce' will turn it to not a time value.
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')

print('FIX APPLEAD')
print(df['start_date'].dtype)

object
FIX APPLEAD
datetime64[ns]


C:\Users\HomePC\AppData\Local\Temp\ipykernel_3272\4138735303.py:8: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['end_date'] = pd.to_datetime(df['end_date'], dayfirst=True, errors='coerce')


## STEP 6: LOGICAL INTEGRITY (CLICKS vs IMPRESSIONS)

Clicks can't be higher than impressions

In [9]:
# There are duplicate columns so this removes the duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

In [10]:
# STEP 6: LOGICAL INTEGRITY (CLICKS vs IMPRESSIONS)

# Defining a mask to see 
impossible_mask = df['clicks'] > df['impressions'] # There shouldn't be instances where clicks should be higher than impressions.
print(df.loc[impossible_mask,['campaign_id','impressions','clicks']].head(3))

Empty DataFrame
Columns: [campaign_id, impressions, clicks]
Index: []


In [11]:
df # Duplicate column is removed

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,True,FA
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,True,EM
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA
...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,True,GO
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,True,IN
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA


## STEP 7: LOGICAL INTEGRITY (TIME TRAVEL)

Cronological order between start and end date. There shouldn'y be any instatces where the end date is in the past compared to the start date.

In [12]:
# STEP 7: LOGICAL INTEGRITY (TIME TRAVEL)

# Define 'time_travel_mask' as less
time_travel_mask = df['end_date'] < df['start_date']
print(df.loc[time_travel_mask, ['campaign_id','start_date','end_date']].head(3)) # Three instances where the end dates are actually before the start dates.

# Making an assumption that the end date is 30 days from the satrt date to fix the issue
df.loc[time_travel_mask, 'end_date'] = df.loc[time_travel_mask, 'start_date'] + pd.Timedelta(days=30)

print('FIX APPLIED')
print(df.loc[time_travel_mask, ['campaign_id','start_date','end_date']].head(3))

   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-05-01
54   CMP-00055 2023-09-01 2023-08-27
71   CMP-00072 2023-02-01 2023-01-27
FIX APPLIED
   campaign_id start_date   end_date
23   CMP-00024 2023-05-06 2023-06-05
54   CMP-00055 2023-09-01 2023-10-01
71   CMP-00072 2023-02-01 2023-03-03


##  HANDLING OUTLIERS (WINSORIZING)

In [13]:
#  HANDLING OUTLIERS (WINSORIZING)
# There are a lot of outliers in the 'spend' column

# Checking the interquantile range of the date to find the middle spread of the data
Q1 = df['spend'].quantile(0.25)
Q3 = df['spend'].quantile(0.75)

# Defining the interquantile range
IQR = Q3 - Q1 # Seeing what is the usaul range/considered like normal variation for money spend in the 'spend' column 

# Defining the upper spend to determine the maximum value to accept in the value spend
upper_limit = Q3 + (3 * IQR) # Getting the 3rd Quarter and adding 3 times the interquantile range

# Checking instances where the spend is higher than the upper limit
outlier_mask = df['spend'] > upper_limit
print(df.loc[outlier_mask,['campaign_id', 'spend']].head(3))

print("FIX APPLIED")
df.loc[outlier_mask, 'spend'] = upper_limit
print(df.loc[outlier_mask,['campaign_id','spend']].head(3))

     campaign_id      spend
789    CMP-00790  500000.00
1443   CMP-01444    8921.51
1460   CMP-01461  500000.00
FIX APPLIED
     campaign_id      spend
789    CMP-00790  8603.5375
1443   CMP-01444  8603.5375
1460   CMP-01461  8603.5375


## STEP 9: STRING PARSING (FEATURE EXTRACTION)

Creating a new column based on an exixting one

In [14]:
# STEP 9: STRING PARSING (FEATURE EXTRACTION)

# Printing the campaign name to extratc the seasons form the 'campaign_name' column
print(df['campaign_name'].head(3))

# Define a new column called 'season'. This basially means that is present after the first underscore and before the second underscore which is the seasons.
df['season'] = df['campaign_name'].str.extract(r'Q\d_([^_]+)_')

print('FIX APPLIED')
print(df[['campaign_name','season']].head(3))

0    Q4_Summer_CMP-00001
1    Q1_Launch_CMP-00002
2    Q3_Winter_CMP-00003
Name: campaign_name, dtype: object
FIX APPLIED
         campaign_name  season
0  Q4_Summer_CMP-00001  Summer
1  Q1_Launch_CMP-00002  Launch
2  Q3_Winter_CMP-00003  Winter


In [15]:
df

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,True,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,True,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaT,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI,Summer
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,True,GO,Summer
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,True,IN,Launch
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA,Winter


## STEP 10 (OPTIONAL): EXPORTING THE ALREADY CLEANED DATASET

Exporting the data into formats so others can use
- to_clipboard
- to_csv
- to_excel
- to_sql
- e.tc.

In [16]:
# Exporting the first five rows of the dataframe to clipboard as CSV format. , is the separator.

df.head().to_clipboard(sep=",") 

### The Output - which is copied to the clipboard

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,True,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,True,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter


In [ ]:
# Exporting the dataframe to a CSV file without the index. to_csv() is the most comonly used method.

df.to_csv('marketing_campaign_data_clean.csv')  # A new .csv file is created in the folder.

In [19]:
# pd.read_csv() is used to read a CSV file into a DataFrame.

pd.read_csv('marketing_campaign_data_clean.csv', index_col=0)  # Reading the CSV file back into a DataFrame. 

,campaign_id,campaign_name,start_date,end_date,channel,impressions,clicks,spend,conversions,active,campaign_tag,season
0,CMP-00001,Q4_Summer_CMP-00001,2023-11-24,2023-12-13,TikTok,16795,197,102.82,20.0,True,TI,Summer
1,CMP-00002,Q1_Launch_CMP-00002,2023-05-06,2023-05-12,Facebook,1860,30,24.33,1.0,True,FA,Launch
2,CMP-00003,Q3_Winter_CMP-00003,2023-12-13,2023-12-20,Email,77820,843,1323.39,51.0,True,EM,Winter
3,CMP-00004,Q1_BlackFriday_CMP-00004,NaN,2023-11-03,TikTok,55886,2019,2180.38,135.0,False,TI,BlackFriday
4,CMP-00005,Q2_Winter_CMP-00005,2023-04-22,2023-04-23,Facebook,7265,169,252.44,30.0,True,FA,Winter
...,...,...,...,...,...,...,...,...,...,...,...,...
2015,CMP-00400,Q3_Summer_CMP-00400,2023-10-31,2023-11-13,TikTok,30592,586,503.95,77.0,True,TI,Summer
2016,CMP-01255,Q4_Summer_CMP-01255,2023-09-01,2023-09-26,Google Ads,20097,897,1641.00,162.0,True,GO,Summer
2017,CMP-01050,Q2_Launch_CMP-01050,2023-02-09,2023-02-21,Instagram,33254,1117,883.82,214.0,True,IN,Launch
2018,CMP-01118,Q4_Winter_CMP-01118,2023-03-30,2023-04-27,Facebook,68728,2960,4198.50,591.0,True,FA,Winter


In [ ]:
# Expoting to Excel

df.to_excel('marketing_campaign_data_clean.xlsx')  # Exporting the dataframe to an Excel file without the index. A new .csv file is created in the folder.

In [ ]:
# To SQL. More advaned use case. Requires a database connection.

from sqlalchemy import create_engine
engine = create_engine('sqlite:///:marketing_campaign_data_clean.db')  # Creating an in-memory SQLite database.

df.to_sql('job_table', con=engine, if_exists='append', index=False)  # Exporting the dataframe to a SQL table named 'job_table'.

In [ ]:
# To parquet, to_parquet() (binary file format, optimized for speed and storage)

df.to_parquet('marketing_campaign_data_clean.parquet')  # Exporting the dataframe to a Parquet file.

In [ ]:
# To pickel to_pickle() (Python-specific binary format)

df.to_pickle('marketing_campaign_data_clean.pkl')  # Exporting the dataframe to a Pickle file.